# 🎲 桌游社活动日报自动生成脚本

本Notebook展示如何：
1. 定时读取目录内的新CSV文件
2. 处理数据并计算关键指标
3. 生成HTML格式的日报报表
4. 模拟定时任务执行

In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
from data_utils import (
    load_data,
    calculate_activity_metrics,
    get_high_risk_activities,
    generate_html_report,
    DEFAULT_PARTICIPATION_THRESHOLD,
    DEFAULT_DURATION_THRESHOLD,
    DEFAULT_SCORE_THRESHOLD,
)

print("报表生成工具加载完成！")

## 1. 单次报表生成示例

In [ ]:
signups_df, results_df, feedbacks_df = load_data()
metrics_df = calculate_activity_metrics(signups_df, results_df, feedbacks_df)

participation_threshold = DEFAULT_PARTICIPATION_THRESHOLD
duration_threshold = DEFAULT_DURATION_THRESHOLD
score_threshold = DEFAULT_SCORE_THRESHOLD

high_risk_df = get_high_risk_activities(
    signups_df,
    results_df,
    feedbacks_df,
    participation_threshold,
    duration_threshold,
    score_threshold
)

print(f"活动总数: {len(metrics_df)}")
print(f"高风险活动数: {len(high_risk_df)}")
print(f"平均参与率: {metrics_df['参与率'].mean()*100:.1f}%")
print(f"平均评分: {metrics_df['平均评分'].mean():.2f}")

In [ ]:
report_path = generate_html_report(metrics_df, high_risk_df, 'daily_report.html')
print(f"HTML报表已生成: {report_path}")

## 2. 高风险活动清单

In [ ]:
high_risk_df[['日期', '活动名', '参与人数', '参与率', '平均时长', '平均评分', '风险类型']].head(10)

## 3. 定时任务模拟

以下代码展示如何设置定时任务，每天自动生成报表。

In [ ]:
def generate_daily_report():
    """生成每日活动报表"""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    report_filename = f'reports/daily_report_{timestamp}.html'

    os.makedirs('reports', exist_ok=True)

    signups_df, results_df, feedbacks_df = load_data()
    metrics_df = calculate_activity_metrics(signups_df, results_df, feedbacks_df)

    high_risk_df = get_high_risk_activities(
        signups_df,
        results_df,
        feedbacks_df,
        participation_threshold=DEFAULT_PARTICIPATION_THRESHOLD,
        duration_threshold=DEFAULT_DURATION_THRESHOLD,
        score_threshold=DEFAULT_SCORE_THRESHOLD
    )

    report_path = generate_html_report(metrics_df, high_risk_df, report_filename)

    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 报表已生成: {report_path}")
    return report_path

### 立即执行一次报表生成

In [ ]:
report_path = generate_daily_report()
print(f"报表路径: {report_path}")

## 4. 监控目录新文件

模拟监控data目录，当有新CSV文件时自动处理。

In [ ]:
def watch_data_directory():
    """监控数据目录，检测新文件"""
    data_dir = 'data'
    processed_files = set()

    current_files = set(os.listdir(data_dir))
    new_files = current_files - processed_files

    if new_files:
        print(f"检测到新文件: {new_files}")
        generate_daily_report()
        processed_files.update(new_files)
    else:
        print("暂无新文件")

    return current_files

In [ ]:
files = watch_data_directory()
print(f"当前数据文件: {files}")

## 5. 批量报表生成

生成历史日期范围内的所有报表。

In [ ]:
def generate_historical_reports(start_date, end_date):
    """生成历史报表"""
    signups_df, results_df, feedbacks_df = load_data()

    date_range = pd.date_range(start=start_date, end=end_date)

    for date in date_range:
        day_signups = signups_df[signups_df['日期'].dt.date == date.date()]
        day_results = results_df[results_df['日期'].dt.date == date.date()]
        day_feedbacks = feedbacks_df[feedbacks_df['日期'].dt.date == date.date()]

        if len(day_signups) > 0:
            day_metrics = calculate_activity_metrics(day_signups, day_results, day_feedbacks)
            day_high_risk = get_high_risk_activities(
                day_signups, day_results, day_feedbacks,
                participation_threshold=DEFAULT_PARTICIPATION_THRESHOLD,
                duration_threshold=DEFAULT_DURATION_THRESHOLD,
                score_threshold=DEFAULT_SCORE_THRESHOLD
            )

            report_filename = f'reports/historical_report_{date.strftime("%Y%m%d")}.html'
            os.makedirs('reports', exist_ok=True)
            generate_html_report(day_metrics, day_high_risk, report_filename)
            print(f"生成报表: {report_filename}")

    print("历史报表生成完成！")

In [ ]:
generate_historical_reports('2026-01-01', '2026-01-10')

## 6. 报表发送通知（示例）

实际应用中可集成邮件、企业微信等通知方式。

In [ ]:
def send_report_notification(report_path, high_risk_count):
    """发送报表通知"""
    print(f"📢 通知: 新报表已生成")
    print(f"   文件: {report_path}")
    print(f"   高风险活动: {high_risk_count} 场")

    if high_risk_count > 5:
        print("⚠️ 警告: 高风险活动数量较多，请关注！")

    print("-" * 50)

In [ ]:
signups_df, results_df, feedbacks_df = load_data()
metrics_df = calculate_activity_metrics(signups_df, results_df, feedbacks_df)
high_risk_df = get_high_risk_activities(
    signups_df, results_df, feedbacks_df,
    DEFAULT_PARTICIPATION_THRESHOLD,
    DEFAULT_DURATION_THRESHOLD,
    DEFAULT_SCORE_THRESHOLD
)
report_path = generate_html_report(metrics_df, high_risk_df, 'notification_test.html')

send_report_notification(report_path, len(high_risk_df))